In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name in {'Baseline_Training', 'CTC_models', 'PhoneticFeatures_Training'}:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from project_paths import (
    BASELINE_MODELS_DIR,
    CTC_MODELS_DIR,
    DEFAULT_BASELINE_MODEL,
    DEFAULT_CTC_ATTENTION_MODEL,
    DEFAULT_CTC_MODEL,
    DEFAULT_PHONETIC_MODEL_2CLASS,
    DEFAULT_PHONETIC_MODEL_3CLASS,
    EXAMPLE_EMBEDDINGS,
    EXAMPLE_PHONEMES,
    EXAMPLE_SEG_B2,
    EXAMPLE_SEG_B4,
    EXAMPLE_WAV,
    PHONEME_CHOICES_SUMMARY,
    PHONETIC_MODELS_DIR,
    TABLES_DIR,
)


In [ ]:
import random
import torchaudio
from torchaudio import functional as FA
import torch
import matplotlib.pyplot as plt
import numpy as np
import torch.nn as nn
import seaborn as sns
from evaluation_tools import _accuracy_end, _accuracy_start, _accuracy_centre, _accuracy_ordered, _accuracy_unordered, _accuracy_centre_flex, _accuracy_ordered_flex_centre
import torch
from content_manager.vencoder.HubertSoft import HubertSoft
from DatasetsModels import PhonemeEmbeddingDataset, ClassificationModel, collate_fn
from GetData import load_selected_embeddings
from torch.utils.data import DataLoader
from VizulalisationTools import chosen_phonemes_stats
from main import train, test, evaluate
import random
from SaveTools import save_experiment_info
import os

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
#content_encoder = HubertSoft(device=device)

In [ ]:
save_path = str(PROJECT_ROOT)

# Модель

In [ ]:
model_id = 'RGCAIQYZHB' #'VACXUXVEXO'
get_embbs = True


In [ ]:
#model = torch.load(r'C:\Users\ext-ananeva\phon_clust\gitlab\ssl_phoneme_clusterizer\Main_project\KXBUVSLLBA_model.pth')

## Загрузить обученную модель

In [ ]:
model_path = os.path.join(save_path, model_id + '_model.pth')
#model_path= '\\'.join(model_path.split(r'\\'))
model_path.split(r'\\')

In [ ]:
model = torch.load(os.path.join(save_path, model_id + '_model.pth'))
model = model.to(device)

# Входные данные

Список целевых звуков

In [ ]:
phoneme_list = ['sil', 'a0', 'a4', 'b', "b'", 'c', 'ch', 'd', "d'", 'e0', 'f', "f'", 'g', 'h', 'i0','i4', 'j', 'k', "k'", 'l', "l'", 'm', "m'", 'n', "n'", 'o0', 'p', "p'", 'r', "r'", 's', "s'", 'sh', 't', "t'", 'u0', 'v', "v'", 'y0', 'z', "z'", 'zh']
phoneme_list = ['a0', 'a4', 'b', 'd','e0', 'f',  'g', 'h', 'i0','i4', 'j', 'k','l', 'm',  'n', 'o0', 'p', 'r', 's', 'sh', 't', 'u0', 'v',  'y0', 'z', 'zh', 'sil']
phoneme_list = ['a0', 'a4', 'b', "b'", 'c', 'ch', 'd', "d'", 'e0', 'f', "f'", 'g', 'h', 'i0','i4', 'j', 'k', "k'", 'l', "l'", 'm', "m'", 'n', "n'", 'o0', 'p', "p'", 'r', "r'", 's', "s'", 'sh', 't', "t'", 'u0', 'v', "v'", 'y0', 'z', "z'", 'zh', 'sil']
phoneme_list = ['a0', 'a4', 'b', "b'", 'c', 'ch', 'd', "d'", 'e0', 'f', "f'", 'g', 'h', 'i0','i4', 'j', 'k', "k'", 'l', "l'", 'm', "m'", 'n', "n'", 'o0', 'p', "p'", 'r', "r'", 's', "s'", 'sh', 't', "t'", 'u0', 'v', "v'", 'y0', 'z', "z'", 'zh', 'sil', 'a1', 'i1','u1', 'y1', ]

num_classes = len(phoneme_list)

Входные файлы с разметкой и аудио

In [ ]:
embs_file = str(EXAMPLE_EMBEDDINGS)
label_file = str(EXAMPLE_PHONEMES)
wav_file = str(EXAMPLE_WAV)

Извлекаем эмбеддинги из аудио

In [ ]:
if get_embbs:
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    content_encoder = HubertSoft(device=device)
    wave, sr = torchaudio.load(wav_file)
    res_wave = FA.resample(wave, sr, 16000)
    
    ext_emb = content_encoder.encoder(res_wave[0].to(device))
    normed_emb = ext_emb[0] / (ext_emb[0] ** 2).sum(0, keepdims=True) ** 0.5
    normed_emb = normed_emb.cpu().numpy()
    
    np.save(EXAMPLE_EMBEDDINGS, normed_emb)
    normed_emb = np.load(EXAMPLE_EMBEDDINGS)
    normed_emb = torch.tensor(normed_emb, dtype=torch.float32)
    normed_emb = normed_emb.T
else:
    #Прсото подгрузим готовые
    normed_emb = np.load(EXAMPLE_EMBEDDINGS)
    normed_emb = torch.tensor(normed_emb, dtype=torch.float32)
    normed_emb = normed_emb.T

normed_emb.shape
    

In [ ]:
model.eval()

# Запуск модели

In [ ]:
prob_distributions = {'start': [], 'centre': [], 'end': []}
y_pred = {'start': [], 'centre': [], 'end': []}
result = []

In [ ]:
with torch.no_grad():
    start = []
    centre = []
    end = []
    for embs in normed_emb:
        embs = embs.unsqueeze(0).to(device)
        start_out, centre_out, end_out = model(embs)

        # Apply softmax to get probabilities
        start_probs = torch.softmax(start_out, dim=1)
        centre_probs = torch.softmax(centre_out, dim=1)
        end_probs = torch.softmax(end_out, dim=1)

        # Get the probability distributions
        start_dist = {phoneme: start_probs[0][i].item() for i, phoneme in enumerate(phoneme_list)}
        start_dist = sorted(start_dist.items(), key=lambda x: x[1], reverse=True)
        centre_dist = {phoneme: centre_probs[0][i].item() for i, phoneme in enumerate(phoneme_list)}
        centre_dist = sorted(centre_dist.items(), key=lambda x: x[1], reverse=True)
        end_dist = {phoneme: end_probs[0][i].item() for i, phoneme in enumerate(phoneme_list)}
        end_dist = sorted(end_dist.items(), key=lambda x: x[1], reverse=True)

        # Store the distributions
        prob_distributions['start'].append(start_dist)
        prob_distributions['centre'].append(centre_dist)
        prob_distributions['end'].append(end_dist)

        # Get the predicted classes (original functionality)
        pred_start = torch.argmax(start_out, dim=1)
        start.append(phoneme_list[pred_start[0]])
        pred_centre = torch.argmax(centre_out, dim=1)
        centre.append(phoneme_list[pred_centre[0]])
        pred_end = torch.argmax(end_out, dim=1)
        end.append(phoneme_list[pred_end[0]])

        preds = []
        preds.append(pred_start.cpu().numpy().tolist())
        preds.append(pred_centre.cpu().numpy().tolist())
        preds.append(pred_end.cpu().numpy().tolist())
        res = [phoneme_list[preds[i][0]] for i in range(len(preds))]
        res.append('')
        result += res

    y_pred['start'] = start
    y_pred['centre'] = centre
    y_pred['end'] = end

# Результаты

## Предсказанные метки

In [ ]:
print('Predictions as phoneme sequences:')
print('_'.join(result))


Предсказания по позициям

In [ ]:
print("\nPredicted classes:")
print("Start:", y_pred['start'])
print("Centre:", y_pred['centre'])
print("End:", y_pred['end'])

## Оценка качества

In [ ]:
lines = open(label_file, 'r').readlines()
y_true = {'start': [],
          'centre': [],
          'end': []}
for line in lines:
    start, centre, end = line.strip().split('_')
    y_true['start'].append(start)
    y_true['centre'].append(centre)
    y_true['end'].append(end)
    
print(y_true)

In [ ]:

print(y_true)
print(y_pred)

In [ ]:
print("\nAccuracy metrics:")
print("accuracy start: " + str(_accuracy_start(y_true, y_pred)))
print("accuracy centre: " + str(_accuracy_centre(y_true, y_pred)))
print("accuracy end: " + str(_accuracy_end(y_true, y_pred)))
print("accuracy ordered: " + str(_accuracy_ordered(y_true, y_pred)))
print("accuracy unordered: " + str(_accuracy_unordered(y_true, y_pred)))
print("accuracy unordered flexible centre: " + str(_accuracy_centre_flex(y_true, y_pred)))
print("accuracy ordered flexible centre: " + str(_accuracy_ordered_flex_centre(y_true, y_pred)))


## Распределения вероятностей в каждом фрейме

In [ ]:
print("\nProbability distributions for each frame:")
for i in range(len(prob_distributions['start'])):
    print(f"\nFrame {i + 1}:")
    print("Start probabilities:", prob_distributions['start'][i])
    print("Centre probabilities:", prob_distributions['centre'][i])
    print("End probabilities:", prob_distributions['end'][i])

### Визуализация

In [ ]:
threshold = 0.01

max_examples = 50

palette = sns.color_palette("tab20", len(phoneme_list))
phoneme_colors = {phoneme: palette[i % len(palette)] for i, phoneme in enumerate(phoneme_list)}

fig, ax = plt.subplots(figsize=(15, 6))

bar_width = 0.2
inner_gap = 0.05
group_gap = 0.5

x_ticks = []
x_labels = []

x_pos = 0

for ex_idx in range(min(len(prob_distributions['start']), max_examples)):
#for ex_idx in range(26, 39):
    for j, part in enumerate(['start', 'centre', 'end']):
        dist = prob_distributions[part][ex_idx]
        dist = [(p, v) for p, v in dist if v > threshold]
        total = sum(v for _, v in dist)
        dist = [(p, v / total) for p, v in dist if total > 0]

        bottom = 0
        for phoneme, prob in dist:
            ax.bar(x_pos, prob, width=bar_width, bottom=bottom,
                   color=phoneme_colors[phoneme], label=phoneme)
            bottom += prob

        x_ticks.append(x_pos)
        x_labels.append(f'{ex_idx}_{part[0]}')

        x_pos += bar_width + inner_gap

    x_pos += group_gap

handles, labels = ax.get_legend_handles_labels()
unique = dict(zip(labels, handles))
ax.legend(unique.values(), unique.keys(), bbox_to_anchor=(1.05, 1), loc='upper left')

ax.set_xticks(x_ticks)
ax.set_xticklabels(x_labels, rotation=45)
ax.set_ylabel("Normalized Probability")
ax.set_title("Phoneme Distributions per Prediction (Start / Centre / End)")

plt.tight_layout()
plt.show()

In [ ]:
from VizulalisationTools import plot_triplet_distribution
plot_triplet_distribution(prob_distributions, phoneme_list)

In [ ]:

def get_phrase(list_of_triplets):
    prev_let = None
    phrase = ''
    phrase_list = []
    for letter in list_of_triplets:
        if letter != prev_let:
            phrase += letter
            prev_let = letter
            phrase_list.append(letter)
    return phrase, phrase_list

print(get_phrase(y_true['centre']))
print(get_phrase(y_pred['centre']))

In [ ]:
prob_distributions

In [ ]:
from Decoding_tools import viterbi_decode, viterbi_lookahead, write_to_seg

In [ ]:
def normalize_prob_dist(prob_dist, eps=1e-8):
    out = []
    for frame in prob_dist:
        frame_dict = {ph: max(p, eps) for ph, p in frame}
        out.append(frame_dict)
    return out
probability_dist = {
    "start": normalize_prob_dist(prob_distributions["start"]),
    "center": normalize_prob_dist(prob_distributions["centre"]),
    "end": normalize_prob_dist(prob_distributions["end"]),
}

In [ ]:
phonemes,phonemes_no_rep, boundaries = viterbi_decode(probability_dist)
phonemes_la,phonemes_no_rep_la, boundaries_la = viterbi_lookahead(probability_dist)


In [ ]:
print(''.join(phonemes_no_rep))
print(''.join(phonemes_no_rep_la))

In [ ]:
res_name = str(EXAMPLE_SEG_B4)
boundaries = [0] + boundaries

pred_labelling = write_to_seg(res_name, boundaries, phonemes_no_rep)
pred_labelling_la = write_to_seg(res_name, boundaries_la, phonemes_no_rep_la)


In [ ]:
len(boundaries)

In [ ]:
target_labelling = []
with open(EXAMPLE_SEG_B2, 'r') as f:
    lines = f.readlines()
    sf = float(lines[1].split('=')[1])
    for i, l in enumerate(lines[7:]):
        start, level, label = l.split(',')
        s = float(start)/(sf*2)
        target_labelling.append((s, label.rstrip('\n')))
target_labelling     

In [ ]:
def phonemes_with_ends(starts, labels, total_duration):
    result = []
    for i, (s, lab) in enumerate(zip(starts, labels)):
        e = starts[i + 1] if i + 1 < len(starts) else total_duration
        result.append((s, e, lab))
    return result


In [ ]:
waveform, sr = torchaudio.load(str(EXAMPLE_WAV))
if waveform.shape[0] > 1:
    waveform = waveform.mean(dim=0)

y = waveform.numpy()[0]
duration = y.shape[0] / sr

In [ ]:
from VizulalisationTools import plot_waveform_with_vlines_dual_labels, plot_waveform_with_labels
import plotly.graph_objects as go

In [ ]:
plot_waveform_with_vlines_dual_labels(y, sr, target_labelling, pred_labelling_la)

In [ ]:
plot_waveform_with_vlines_dual_labels(y, sr, target_labelling, pred_labelling)

In [ ]:
pred_labelling_la

In [ ]:
probability_dist

In [ ]:
phonemes,phonemes_no_rep, boundaries

In [ ]:
for b in boundaries:
    print(list(probability_dist['start'][b].keys())[0], list(probability_dist['center'][b].keys())[0], list(probability_dist['end'][b].keys())[0])

In [ ]:
for b in range(180):
    print(list(probability_dist['start'][b].keys())[0], list(probability_dist['center'][b].keys())[0], list(probability_dist['end'][b].keys())[0])

In [ ]:
target_list = [y_true['centre'][0]]
for ph in y_true['centre'][1:]:
    if ph!=target_list[-1]:
        target_list.append(ph)
    
target_list

In [ ]:
from Levenshtein import distance
def cer(true, pred):
    return distance(true, pred) / len(true) if true else 0.0

pred_list = [pair[1] for pair in pred_labelling]
cer = cer(target_list, pred_list)


In [ ]:
cer

In [ ]:
from Levenshtein import distance
def cer(true, pred):
    return distance(true, pred) / len(true) if true else 0.0

pred_list = [pair[1] for pair in pred_labelling_la]
res = cer(target_list, pred_list)
res

In [ ]:
true_phrase, pred_phrase = get_phrase(y_true['centre'])
res = cer(true_phrase, pred_phrase)
print(res)